# DataFog PII-NER v1 — Local Training (RTX 3090)

Local adaptation of the full training notebook for an RTX 3090 (24GB VRAM).

**Key differences from Colab A100 notebook:**
- Batch size 8 with 4x gradient accumulation (effective batch = 32)
- FP16 mixed precision (3090 lacks native BF16 support)
- Local paths, no git clone step
- Estimated training time: ~8-12 hours for 10 epochs

## 1. Setup

In [ ]:
import sys, os

# Add source to path (local dev install)
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import datafog_pii_ner
print(f"datafog_pii_ner loaded from: {datafog_pii_ner.__file__}")
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    mem = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f"Memory: {mem / 1e9:.1f} GB")
    # Check compute capability for FP16/BF16 support
    cc = f"{props.major}.{props.minor}"
    print(f"Compute capability: {cc}")
    if props.major >= 8:
        print("BF16: supported")
    else:
        print("BF16: not supported (using FP16)")
else:
    raise RuntimeError("No GPU found")

In [ ]:
# WandB login (optional — set to "none" below to skip)
import wandb
wandb.login()

## 2. Configuration

In [ ]:
CONFIG = {
    # Model
    "backbone": "microsoft/deberta-v3-xsmall",
    "max_seq_len": 256,
    "max_char_len": 20,
    "dropout": 0.1,

    # Training — RTX 3090 (24GB)
    # Effective batch = 8 * 4 = 32 (matches A100 config)
    "epochs": 10,
    "batch_size": 8,
    "gradient_accumulation_steps": 4,
    "lr_backbone": 2e-5,
    "lr_head": 1e-3,
    "warmup_ratio": 0.1,
    "weight_decay": 0.01,

    # FP16 for RTX 3090 (no native BF16)
    # CRF head runs in FP32 via fused.float() + autocast(enabled=False)
    "fp16": True,
    "bf16": False,

    # Data
    "val_ratio": 0.1,
    "test_ratio": 0.1,
    "seed": 42,

    # Output — local paths
    "output_dir": os.path.join(PROJECT_ROOT, "output"),
    "run_name": "pii-ner-v1-local",
}

effective_batch = CONFIG["batch_size"] * CONFIG["gradient_accumulation_steps"]
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Batch size: {CONFIG['batch_size']} x {CONFIG['gradient_accumulation_steps']} = {effective_batch} effective")
print(f"Mixed precision: FP16")
print(f"Output: {CONFIG['output_dir']}")

## 2.5 Preflight Check

Run training steps on synthetic data at **full batch size** to verify the optimizer + FP16 pipeline is numerically stable. Explicitly checks for NaN in loss and model weights (HF Trainer v5.0 silently renders NaN as 0.000000).

In [ ]:
import math
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, TrainingArguments
from datafog_pii_ner.data.collator import PiiDataCollator
from datafog_pii_ner.data.label_schema import NUM_LABELS
from datafog_pii_ner.model.pii_model import PiiNerConfig, PiiNerModel
from datafog_pii_ner.training.train import PiiTrainer

print("=== PREFLIGHT CHECK ===")
print(f"Testing: batch_size={CONFIG['batch_size']}, fp16={CONFIG['fp16']}, bf16={CONFIG['bf16']}")

_tokenizer = AutoTokenizer.from_pretrained(CONFIG["backbone"])
_seq_len = CONFIG["max_seq_len"]  # Use real sequence length, not short
_n_samples = CONFIG["batch_size"] * 2  # Enough for 2 full batches

# Synthetic data at full sequence length and batch size
_fake_data = {
    "input_ids": np.random.randint(1, 1000, (_n_samples, _seq_len)).tolist(),
    "attention_mask": np.ones((_n_samples, _seq_len), dtype=int).tolist(),
    "labels": np.random.randint(0, NUM_LABELS, (_n_samples, _seq_len)).tolist(),
    "char_ids": np.random.randint(0, 100, (_n_samples, _seq_len, CONFIG["max_char_len"])).tolist(),
}
_ds = Dataset.from_dict(_fake_data)
_collator = PiiDataCollator(tokenizer=_tokenizer, max_char_len=CONFIG["max_char_len"])

_config = PiiNerConfig(backbone=CONFIG["backbone"], num_labels=NUM_LABELS)
_model = PiiNerModel(_config)

_args = TrainingArguments(
    output_dir=os.path.join(CONFIG["output_dir"], "preflight_check"),
    num_train_epochs=1,
    per_device_train_batch_size=CONFIG["batch_size"],
    learning_rate=CONFIG["lr_backbone"],
    fp16=CONFIG["fp16"],
    bf16=CONFIG["bf16"],
    report_to="none",
    logging_steps=1,
    max_steps=5,  # 5 steps to catch delayed NaN
    remove_unused_columns=False,
    save_strategy="no",
)

_trainer = PiiTrainer(
    model=_model,
    args=_args,
    train_dataset=_ds,
    data_collator=_collator,
    lr_backbone=CONFIG["lr_backbone"],
    lr_head=CONFIG["lr_head"],
)

_result = _trainer.train()
_loss = _result.training_loss

# === NaN checks ===
checks_passed = True

# Check 1: Training loss is a real number
if math.isnan(_loss) or math.isinf(_loss):
    print(f"FAIL: Training loss is {_loss}")
    checks_passed = False
else:
    print(f"PASS: Training loss = {_loss:.4f}")

# Check 2: Model weights contain no NaN
_nan_params = []
for name, param in _model.named_parameters():
    if torch.isnan(param).any():
        _nan_params.append(name)
if _nan_params:
    print(f"FAIL: NaN in {len(_nan_params)} parameter tensors:")
    for p in _nan_params[:5]:
        print(f"  - {p}")
    checks_passed = False
else:
    print(f"PASS: All {sum(1 for _ in _model.parameters())} parameter tensors are finite")

# Check 3: Loss decreased (model is actually learning, not stuck)
_log_history = _trainer.state.log_history
_losses = [h["loss"] for h in _log_history if "loss" in h]
if len(_losses) >= 2:
    if any(math.isnan(l) for l in _losses):
        print(f"FAIL: NaN in step losses: {_losses}")
        checks_passed = False
    elif _losses[-1] < _losses[0]:
        print(f"PASS: Loss decreasing ({_losses[0]:.1f} → {_losses[-1]:.1f})")
    else:
        print(f"WARN: Loss not decreasing ({_losses[0]:.1f} → {_losses[-1]:.1f}) — may be OK for 5 steps")

if checks_passed:
    print("\n=== PREFLIGHT PASSED — safe to proceed ===")
else:
    raise RuntimeError(
        "PREFLIGHT FAILED — do not proceed with training. "
        "Check mixed precision settings and optimizer config."
    )

del _model, _trainer, _ds, _collator, _args, _result
import gc; gc.collect()
torch.cuda.empty_cache()

## 3. Load Data

In [ ]:
from transformers import AutoTokenizer
from datafog_pii_ner.data.dataset import load_pii_datasets
from datafog_pii_ner.data.label_schema import NUM_LABELS

tokenizer = AutoTokenizer.from_pretrained(CONFIG["backbone"])

print("Loading all datasets (this may take a few minutes)...")
datasets = load_pii_datasets(
    tokenizer=tokenizer,
    max_seq_len=CONFIG["max_seq_len"],
    max_char_len=CONFIG["max_char_len"],
    val_ratio=CONFIG["val_ratio"],
    test_ratio=CONFIG["test_ratio"],
    seed=CONFIG["seed"],
)

print(f"\nDataset sizes:")
print(f"  Train:      {len(datasets['train']):,}")
print(f"  Validation: {len(datasets['validation']):,}")
print(f"  Test:       {len(datasets['test']):,}")
print(f"  Total:      {sum(len(datasets[s]) for s in datasets):,}")
print(f"  Labels:     {NUM_LABELS}")

## 4. Initialize Model

In [ ]:
from datafog_pii_ner.model.pii_model import PiiNerConfig, PiiNerModel

config = PiiNerConfig(
    backbone=CONFIG["backbone"],
    num_labels=NUM_LABELS,
    dropout=CONFIG["dropout"],
)
model = PiiNerModel(config)

param_count = sum(p.numel() for p in model.parameters())
trainable_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {param_count:,}")
print(f"Trainable parameters: {trainable_count:,}")

# Estimate VRAM usage
param_mb = param_count * 4 / 1e6  # FP32 weights
grad_mb = param_mb  # gradients
opt_mb = param_mb * 2  # AdamW states (m, v)
print(f"\nEstimated VRAM: {param_mb + grad_mb + opt_mb:.0f} MB model/optimizer")
print(f"  + activations (batch-dependent, ~2-6 GB for batch_size=8)")
print(f"  RTX 3090 has 24 GB — should fit comfortably")

## 5. Train

In [ ]:
from transformers import TrainingArguments
from datafog_pii_ner.data.collator import PiiDataCollator
from datafog_pii_ner.training.metrics import compute_metrics
from datafog_pii_ner.training.train import PiiTrainer

collator = PiiDataCollator(tokenizer=tokenizer, max_char_len=CONFIG["max_char_len"])

training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["lr_backbone"],
    warmup_ratio=CONFIG["warmup_ratio"],
    weight_decay=CONFIG["weight_decay"],
    fp16=CONFIG["fp16"],
    bf16=CONFIG["bf16"],
    eval_strategy="epoch",
    save_strategy="epoch",
    metric_for_best_model="overall_f1",
    load_best_model_at_end=True,
    report_to="wandb",
    run_name=CONFIG["run_name"],
    logging_steps=50,
    remove_unused_columns=False,
    dataloader_num_workers=4,
    save_total_limit=3,
)

print(f"Backbone LR: {CONFIG['lr_backbone']}, Head LR: {CONFIG['lr_head']}")
print(f"Mixed precision: FP16 (CRF head runs in FP32 via autocast bypass)")

# PiiTrainer uses eps=1.0 for backbone to prevent AdamW NaN
# (see smoke_test_walkthrough.md for details)
trainer = PiiTrainer(
    model=model,
    args=training_args,
    train_dataset=datasets["train"],
    eval_dataset=datasets["validation"],
    data_collator=collator,
    compute_metrics=compute_metrics,
    lr_backbone=CONFIG["lr_backbone"],
    lr_head=CONFIG["lr_head"],
)

print(f"\nStarting training for {CONFIG['epochs']} epochs...")
train_result = trainer.train()
print(f"\nTraining complete. Final loss: {train_result.training_loss:.4f}")

## 6. Evaluate on Test Set

In [ ]:
print("Evaluating on held-out test set...")
test_results = trainer.evaluate(datasets["test"])

print("=" * 60)
print("TEST SET RESULTS")
print("=" * 60)
print(f"  Overall F1:        {test_results.get('eval_overall_f1', 0):.4f}")
print(f"  Overall Precision: {test_results.get('eval_overall_precision', 0):.4f}")
print(f"  Overall Recall:    {test_results.get('eval_overall_recall', 0):.4f}")
print()

for tier in [1, 2, 3, 4]:
    key = f"eval_tier_{tier}_recall"
    if key in test_results:
        target = {1: 0.98, 2: 0.95, 3: 0.90, 4: 0.85}[tier]
        actual = test_results[key]
        status = "PASS" if actual >= target else "FAIL"
        print(f"  Tier {tier} recall: {actual:.4f}  (target >= {target})  [{status}]")

print("\nPer-entity F1 (top 20):")
type_metrics = [(k, v) for k, v in test_results.items() if k.startswith("eval_type_") and k.endswith("_f1")]
type_metrics.sort(key=lambda x: x[1], reverse=True)
for k, v in type_metrics[:20]:
    name = k.replace("eval_type_", "").replace("_f1", "")
    print(f"  {name:30s} {v:.4f}")

## 7. Save Model

In [ ]:
import json

save_path = os.path.join(CONFIG["output_dir"], "best_model")
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model saved to {save_path}")

with open(os.path.join(save_path, "training_config.json"), "w") as f:
    json.dump(CONFIG, f, indent=2, default=str)
print(f"Training config saved")

with open(os.path.join(save_path, "test_results.json"), "w") as f:
    json.dump({k: float(v) if hasattr(v, '__float__') else v for k, v in test_results.items()}, f, indent=2)
print(f"Test results saved")